## 0 · One Kick, One Question

> **Match day. 20 metres from goal. One free kick.**
>
> Forget calculus for a moment. You already know how to improve this shot:
>
> 1. **Choose** an angle.
> 2. **Watch** what the ball does.
> 3. **Name the miss:** wall, over the bar, or wrong part of the net.
> 4. **Change the cause** in the direction that should reduce that miss.
> 5. Repeat until the correction becomes tiny.
>
> That loop is gradient descent.

The ball's centre must clear the 1.8 m wall, pass fully inside the 2.44 m goal frame, and arrive near a 1.10 m target at the back of the net. The animation below keeps the match visible while the hidden quantities change.

**Do not study the numbers yet. Watch their story:** one choice at the foot becomes a flight, the flight becomes an impact, and the impact becomes feedback.

![Cinematic forward pass of the free kick with live motion and a restrained telemetry replay](images/free-kick-forward-telemetry.gif)

**What to notice:** the angle is chosen once; everything after it is a consequence. Calculus gives us a disciplined way to connect the final consequence back to that first choice.

<small>Match photograph: Michael Barera, [“Detroit City FC v. San Antonio FC 2023 20 (free kick)”](https://commons.wikimedia.org/wiki/File:Detroit_City_FC_v._San_Antonio_FC_2023_20_(free_kick).jpg), Wikimedia Commons, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Cropped, color-graded, and composited with physics-driven motion and instructional overlays.</small>

# Mathematical Foundations for Machine Learning

## The idea before the vocabulary

A model learns the way this striker improves:

> **Make a choice → observe the result → measure the miss → trace the miss back → adjust the choice.**

In this notebook, the choice is one launch angle. In a neural network, the choices are millions of weights. The number of choices changes; the learning loop does not.

| In the free kick | Plain-language job | Mathematical name |
| --- | --- | --- |
| Point toward something | Compare two directions | Dot product |
| Notice when rising becomes falling | Measure change at this instant | Derivative |
| Turn repeated misses into corrections | Improve a choice step by step | Gradient descent |
| Combine angle, speed, wind, and distance | Mix many signals at once | Matrix multiplication |
| Ask how the impact came from the angle | Pass responsibility backward | Chain rule / backpropagation |
| Allow for imperfect execution | Describe a range of possible outcomes | Probability |

The notebook will always show the physical idea first. Equations appear afterward in collapsible **“Name the pattern”** sections. Open them when the visual story already makes sense.

In [ ]:
# Dependencies
import subprocess, sys

# Only install packages that are not already importable in this environment
required = [("numpy", "numpy"), ("scipy", "scipy")]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

import numpy as np
from scipy import stats

np.random.seed(42)
print("All dependencies ready")

# Physical constants for the free kick scenario
g = 9.81          # gravity (m/s²)
v0 = 20.0        # launch speed (m/s)
BALL_RADIUS = 0.11
WALL_X = 9.15    # wall position (m)
WALL_H = 1.8     # wall height (m)
GOAL_X = 20.0    # front of the goal (m)
CROSS_H = 2.44   # crossbar height (m)
NET_X = 21.5     # back of the net (m)
TARGET_H = 1.10  # intended ball-centre height at back-net impact (m)
GOAL_BOTTOM = BALL_RADIUS
GOAL_TOP = CROSS_H - BALL_RADIUS


def ball_state(x, theta_deg):
    """Return time, height, and velocity when the ball reaches horizontal position x."""
    theta = np.radians(theta_deg)
    vx = v0 * np.cos(theta)
    t = x / vx
    vy = v0 * np.sin(theta) - g * t
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    return {"x": float(x), "t": float(t), "y": float(y), "vx": float(vx), "vy": float(vy)}


def ball_height(x, theta_deg):
    """Height of the ball centre at horizontal position x and launch angle theta (degrees)."""
    return ball_state(x, theta_deg)["y"]


print("\nFree kick setup:")
print(f"  Launch speed: {v0} m/s")
print(f"  Wall at {WALL_X}m, ball centre must clear {WALL_H + BALL_RADIUS:.2f}m")
print(f"  Goal plane at {GOAL_X}m, legal centre window [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m")
print(f"  Back-net target at ({NET_X:.1f}m, {TARGET_H:.2f}m)")

---

## Part 1 — Which Way Are We Pointing?

Before the ball can travel well, the kick direction must agree with the intended direction.

Imagine two arrows laid on top of each other:

- same direction → they strongly agree,
- sideways to each other → they do not help each other,
- opposite directions → one fights the other.

A dot product compresses that directional agreement into one number. You already understand the geometry; the operation only gives it a score.

#### Predict first

The kick points at 25°. The intended direction is 20°. They differ by only 5°.

- **(a)** Almost complete agreement
- **(b)** Half agreement
- **(c)** They oppose each other

Make the visual prediction first, then run the code.

<details>
<summary><strong>Name the pattern: dot product</strong></summary>

For unit vectors, the score is the cosine of the angle between them:

$$\mathbf{a}\cdot\mathbf{b}=\cos(\text{angle between them})$$

The expanded coordinate formula is $a_1b_1+a_2b_2$. A dense layer repeats this weighted-alignment calculation across many inputs.

</details>

> **Intuition first:** Imagine rotating your kick direction toward the goal. At exactly the right angle they point the same way — that's maximum overlap. At 90° they're perpendicular — zero overlap. The dot product captures that overlap as a single number: 1 when perfectly aligned, 0 when perpendicular, -1 when opposite.

**Vector alignment visualised:** The dot product measures how much two vectors point in the same direction.

### Pause the picture in your head

Hold the kick arrow fixed and rotate the wind arrow around it.

| Wind direction | What you would feel | Alignment score |
| --- | --- | --- |
| Behind the kick | A push in the same direction | Large and positive |
| Sideways | Drift, but no forward help | Near zero |
| Into the kick | Resistance | Negative |

The score changes smoothly as the arrow rotates. Nothing magical happens at a particular angle; the number simply tracks how much one direction lies along the other.

#### Say it without notation

> The dot product answers: **“How much of this direction helps that direction?”**

Now run the next short cell. It calculates the score for a 25° kick and a 20° intended direction.

In [ ]:
#  Part 1: Dot products and vector alignment
# Build 2D unit vectors for the actual kick angle and the ideal goal-facing angle
kick_dir = np.array([np.cos(np.radians(25)), np.sin(np.radians(25))])  # 25° angle
goal_dir = np.array([np.cos(np.radians(20)), np.sin(np.radians(20))])  # ideal 20°

# Dot product of two unit vectors equals the cosine of the angle between them
dot = np.dot(kick_dir, goal_dir)
alignment = np.degrees(np.arccos(np.clip(dot, -1, 1)))

print(f"Kick direction (25°): {kick_dir.round(3)}")
print(f"Ideal direction (20°): {goal_dir.round(3)}")
print(f"Dot product: {dot:.4f}")
print(f"Angular difference: {alignment:.1f}°")
print()
print("Dot product in ML: W · x = weighted sum of features")
print("  'How much does each feature contribute to the prediction?'")
print("  A dot product is the core operation in every Dense layer.")

#### What just happened — and what's missing

The dot product between kick (25°) and goal (20°) is very close to 1 — confirming prediction (a). Five degrees of misalignment barely dents the alignment score because cosine is very flat near zero.

**This is the most-used operation in all of ML.** Every `layers.Dense` layer computes $\mathbf{W} \cdot \mathbf{x}$ — a dot product of the weight row with the input — for every output neuron, simultaneously. Attention scores in transformers are dot products of query and key vectors. If you understand "dot product = measure of alignment," you understand the core computation of every model in this curriculum.

**Missing piece:** The dot product tells us _how aligned_ the kick is, but not _when the ball peaks_ — and we need the peak to know whether it clears the wall. For that we need derivatives.

---

#### Your turn — dot product geometry

```python
# CHANGE: try kick_angle = 90 (straight up). What dot product with goal_dir (20°) do you predict?
# Then try kick_angle = 200. What happens when the kick goes backward?
kick_angle = 90   # degrees — change this
goal_angle = 20   # fixed

kick = np.array([np.cos(np.radians(kick_angle)), np.sin(np.radians(kick_angle))])
goal = np.array([np.cos(np.radians(goal_angle)), np.sin(np.radians(goal_angle))])
dp = np.dot(kick, goal)
print(f"kick at {kick_angle}° vs goal at {goal_angle}°  →  dot product = {dp:.4f}")
```

---

## Part 2 — The Instant the Ball Stops Rising

Watch a ball rise in slow motion.

- Early in the flight, each frame is noticeably higher than the last.
- Near the top, the height gained per frame shrinks.
- For one instant, the next frame is neither higher nor lower.
- After that, each frame drops.

That **height gained per step** is the derivative. Positive means rising. Negative means falling. Zero marks the turn.

#### Predict first

For the 25° shot, where does the turn happen?

1. Around the wall at 10 m
2. Between the wall and goal at 15–16 m
3. At the goal line near 20 m

The code samples the flight and then checks the exact turning point.

<details>
<summary><strong>Name the pattern: derivative</strong></summary>

The trajectory has a height function $h(x)$. Its derivative $dh/dx$ means “how many metres of height change for the next metre forward.” At the peak:

$$\frac{dh}{dx}=0$$

The full projectile formula lives in the code. The intuition to keep is simpler: **a derivative is a local change meter**.

</details>

In [ ]:
# Part 2: derivative to find the peak height
theta_deg = 25.0
theta_rad = np.radians(theta_deg)

# Sample the trajectory at 200 points between launch and the back of the net
x_vals = np.linspace(0, NET_X, 200)
heights = [ball_height(x, theta_deg) for x in x_vals]

# The peak is where dh/dx = 0; argmax on dense samples provides a numerical check
peak_idx = np.argmax(heights)
peak_x = x_vals[peak_idx]
peak_h = heights[peak_idx]

# Closed-form peak location from dh/dx = 0
x_peak_analytic = v0**2 * np.sin(2 * theta_rad) / (2 * g)
wall_state = ball_state(WALL_X, theta_deg)
goal_state = ball_state(GOAL_X, theta_deg)
net_state = ball_state(NET_X, theta_deg)

print(f"At launch angle {theta_deg}°:")
print(f"  Sampled peak: x={peak_x:.2f}m, y={peak_h:.2f}m")
print(f"  Analytic peak: x={x_peak_analytic:.2f}m")
print()
print(f"  Ball centre at wall: {wall_state['y']:.2f}m  "
      f"(need > {WALL_H + BALL_RADIUS:.2f}m: {'PASS' if wall_state['y'] > WALL_H + BALL_RADIUS else 'FAIL'})")
print(f"  Ball centre at goal: {goal_state['y']:.2f}m  "
      f"(legal [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m: {'PASS' if GOAL_BOTTOM < goal_state['y'] < GOAL_TOP else 'FAIL'})")
print(f"  Predicted back-net height: {net_state['y']:.2f}m  (target {TARGET_H:.2f}m)")
print()
print("Prediction check: answer 2 — the ball peaks between wall and goal.")
print("The derivative locates the turning point; the constraints tell us whether that curve scores.")

### Freeze the flight at three moments

| Moment | What the ball is doing | What that tells us |
| --- | --- | --- |
| Over the wall | Still rising | Clearing the wall does not guarantee a goal |
| At 15.6 m | Neither rising nor falling | This is the peak: local change is zero |
| At the goal | Falling, but still 3.35 m high | The shot turns downward too late to fit under the bar |

#### Build the causal picture

A higher peak is not automatically better. Moving the peak changes the entire second half of the flight. What matters is the ball's state **at each constraint**, not one impressive number in isolation.

> The derivative helps us locate the turn. The match constraints decide whether the resulting curve is useful.

#### Read the shot like a coach

The 25° ball turns at about **15.6 m**. That tells us the shape of the flight, but not whether it scores.

Now inspect the important moments:

- **At the wall:** 3.02 m — safely over.
- **At the goal plane:** 3.35 m — over the crossbar.
- **Verdict:** good first constraint, failed second constraint.

The useful lesson is not “the derivative solved the kick.” It is:

> **The derivative tells us how behavior is changing. The loss tells us whether that behavior serves the goal.**

To improve the shot, we now ask a practical question: if a slightly steeper angle makes the miss worse, which way should the next angle move?

<details>
<summary><strong>Compact rule</strong></summary>

If increasing the angle increases the loss, then $dL/d\theta>0$. The correction must move in the opposite direction, so the angle decreases.

</details>

#### Your turn

Try 35°, then 15°. Before running the code, say the miss out loud: **wall**, **high**, **low**, or **target**. Classification before calculation builds the right intuition.

---

## Part 3 — Learn by Missing

A 35° kick sails high. What information is hidden in that miss?

- The wall is already safe, so no correction is needed there.
- The ball crosses above the frame, so “higher” is the wrong direction.
- The back-net impact is also high, so the same angle change improves both errors.

We turn those observations into a **wrongness score** called loss. A worse miss gets a larger score. A legal shot near the target gets a score near zero.

Then we run one tiny experiment:

> If I nudge the angle upward, does wrongness rise or fall?

That answer is the gradient. If upward makes things worse, step downward. Repeat.

#### Predict first

Starting from 35°, what should happen?

1. The angle moves down toward ~19°
2. The angle moves up because the gradient is positive
3. The angle stays put because it already clears the wall

Watch each attempted kick below. The faded trails are memory: every miss becomes evidence for the next correction.

<details>
<summary><strong>Name the pattern: loss, gradient, update</strong></summary>

The loss combines four plain penalties: wall collision, below-frame miss, above-frame miss, and distance from the back-net target. Only violated constraints contribute.

The update is:

$$\text{new angle}=\text{old angle}-\text{learning rate}\times\text{gradient}$$

The minus sign means “move opposite the direction that makes wrongness grow.”

</details>

![Cinematic sequence of repeated free kicks where each visible miss produces the next angle correction](images/gradient-descent-free-kick.gif)

**Follow the verbs:** shoot → judge → correct. The numbers confirm the story; they are not the story.

In [ ]:
# Part 3: gradient descent on the physically complete free-kick loss
def kick_loss(theta_deg):
    """Wall + goal-frame constraints plus back-net target error; lower is better."""
    h_wall = ball_height(WALL_X, theta_deg)
    h_goal = ball_height(GOAL_X, theta_deg)
    h_net = ball_height(NET_X, theta_deg)

    wall_gap = max(0.0, WALL_H + BALL_RADIUS - h_wall)
    goal_low_gap = max(0.0, GOAL_BOTTOM - h_goal)
    goal_high_gap = max(0.0, h_goal - GOAL_TOP)
    net_error = h_net - TARGET_H

    return wall_gap**2 + 2 * goal_low_gap**2 + 2 * goal_high_gap**2 + 0.25 * net_error**2


learning_rate = 2.0
epsilon = 1e-4
theta = 35.0
history = [theta]

print(f"start: angle={theta:.3f}°  loss={kick_loss(theta):.6f}")

# Estimate the local slope, then move in the opposite direction
for step in range(36):
    gradient = (kick_loss(theta + epsilon) - kick_loss(theta - epsilon)) / (2 * epsilon)
    theta = np.clip(theta - learning_rate * gradient, 8.0, 45.0)
    history.append(float(theta))
    if step < 3 or (step + 1) % 6 == 0:
        print(f"step {step + 1:2d}: angle={theta:7.3f}°  dL/dθ={gradient:+9.5f}  loss={kick_loss(theta):.6f}")

final_angle = history[-1]
final_wall = ball_height(WALL_X, final_angle)
final_goal = ball_height(GOAL_X, final_angle)
final_net = ball_height(NET_X, final_angle)
is_scoreable = (
    final_wall > WALL_H + BALL_RADIUS
    and GOAL_BOTTOM < final_goal < GOAL_TOP
    and abs(final_net - TARGET_H) < 0.08
)

print(f"\nfinal angle: {final_angle:.3f}°")
print(f"wall={final_wall:.3f}m  goal={final_goal:.3f}m  net={final_net:.3f}m")
print(f"final loss={kick_loss(final_angle):.8f}  target hit={is_scoreable}")
print("Prediction check: answer 1 — a positive gradient caused a downward angle update.")

### Read the learning process without a graph

| Attempt | Angle | Wrongness | Coach's read |
| ---: | ---: | ---: | --- |
| 0 | 35.000° | 45.6755 | Far too high — make a large correction down |
| 1 | 21.133° | 0.1208 | Legal direction, still high — keep moving down |
| 6 | 20.189° | 0.0344 | Much closer — use a smaller correction |
| 18 | 19.348° | 0.0017 | Almost centered — correction is now tiny |
| 36 | 19.134° | 0.000018 | Target locked |

Two intuitions matter more than the curve:

1. **Large miss, large useful signal.** The first correction is dramatic because the high shot is unambiguous.
2. **Small miss, small useful signal.** Near the target, the gradient fades, so updates naturally become gentle.

The optimizer is not becoming tired or cautious. The local evidence is simply telling it that less correction remains.

---

#### Your turn — learning rate is a unit-bearing decision

Our parameter is measured in degrees and the gradient in loss per degree, so the learning rate controls how much angular correction one unit of gradient produces.

```python
# CHANGE learning_rate_test to 0.2, then 5.0. Predict first:
#   0.2 -> stable but slow
#   5.0 -> large corrections; clipping may make the path bounce against a bound
learning_rate_test = 0.2
theta_test = 35.0
history_test = [theta_test]

for step in range(36):
    gradient_test = (kick_loss(theta_test + 1e-4) - kick_loss(theta_test - 1e-4)) / 2e-4
    theta_test = np.clip(theta_test - learning_rate_test * gradient_test, 8.0, 45.0)
    history_test.append(float(theta_test))

print(f"lr={learning_rate_test}: angle={theta_test:.3f}°, loss={kick_loss(theta_test):.6f}")
print(f"distance from target solution: {abs(theta_test - final_angle):.3f}°")
```

A learning rate is not "good" in isolation. It is good when its update scale matches the parameterisation and curvature of the current problem.

#### What the optimizer actually knew

Gradient descent changed a visible high miss at 35° into a target-height shot near **19.13°**.

It did **not** know the final answer in advance. At each attempt it knew only:

1. how wrong the current shot was,
2. whether a slightly larger angle made that wrongness rise or fall,
3. how large a correction it was allowed to make.

That is enough.

The important bridge to machine learning is not “a ball rolls down a bowl.” It is:

> **A parameter caused an outcome. A loss judged the outcome. A gradient assigned direction to the cause.**

Our single angle can be tested by nudging it up and down. A neural network has millions of parameters, so repeating that experiment one parameter at a time would be impossibly slow. Part 5 shows how the chain rule reuses one forward pass to send responsibility backward efficiently.

---

## Part 4 — More Than One Knob

So far we changed one knob: angle. A real shot also depends on speed, wind, distance, wall height, and target location.

Imagine a control desk:

- one row of dials decides predicted wall clearance,
- another row decides predicted goal height,
- another could decide scoring confidence.

Each dial says how strongly one input should influence one output. A matrix is simply that whole control desk written as numbers, so all the weighted combinations happen together.

#### Predict first

If every output uses some amount of angle and some amount of speed, what happens when angle changes?

- **(a)** Only one output can change
- **(b)** Every output connected to angle can change
- **(c)** Nothing changes because the bias absorbs it

<details>
<summary><strong>Name the pattern: matrix transformation</strong></summary>

$$\mathbf{y}=W\mathbf{x}+\mathbf{b}$$

- $\mathbf{x}$ is the row of input knobs.
- Each row of $W$ is one output's set of dial strengths.
- $\mathbf{y}$ is the new representation.

The compact notation matters later. For now, read it as: **mix all relevant inputs into every output that needs them**.

</details>

> **Picture it as a mixing desk:** each input feature has a dial leading into each output. The matrix stores all dial positions. Matrix multiplication applies the entire mix in one coordinated operation.

> **Bridge from the kick:** angle and speed are no longer isolated facts. They become inputs whose combination predicts several consequences. Learning means adjusting the mixing dials until those consequences become useful.

In [ ]:
#  Part 4: Matrix as a linear transformation
W = np.array([[2, 0.5], [-0.5, 1.5]])
b = np.array([0.1, -0.2])

features = np.array([0.4, 0.8])  # angle=40% of max, speed=80% of max

# This is exactly what a Dense/Linear layer computes: matrix-vector multiply plus bias
output = W @ features + b

print(f"Input features (angle, speed): {features}")
print(f"Weight matrix W:\n{W}")
print(f"Bias b: {b}")
print(f"Output W@x + b: {output.round(4)}")
print()
print("In a neural network:")
print("  features = pixel values / token embeddings / sensor readings")
print("  W = learned weights (what the model found useful)")
print("  output = the model's internal representation")
print()
print(f"Parameter count: W has {W.size} + b has {b.size} = {W.size + b.size} learnable values")
print("GPT-2's first attention layer: 768×2304 = 1,769,472 parameters (same operation, bigger numbers)")

### Walk one signal through the mixing desk

The runnable cell produced two outputs from two inputs: angle and speed.

Now imagine covering the **speed** column of the weight matrix with your hand. What remains is every route by which angle can affect the outputs. One angle value can push one output up strongly, push another down slightly, or be ignored entirely — depending on the learned dial for each route.

Then cover the **angle** column. You have isolated speed's influence.

This column-by-column view is often more intuitive than staring at the whole matrix:

- **a column follows one input** into every output,
- **a row gathers all inputs** for one output,
- the full multiplication performs every route together.

> A matrix is not an abstract grid. It is a map of who can influence whom, and by how much.

#### What just happened — and what's missing

The matrix **mixed every input feature into every output** — prediction (b). That's not a bug; it's the feature. The matrix learns which combinations of inputs matter for each output.

| Toy (this notebook)        | GPT-2 (production)                    |
| -------------------------- | ------------------------------------- |
| 2 input features           | 768 token embedding dimensions        |
| 2 output features          | 2304 (Q, K, V concatenated)           |
| 4 weight values            | 1,769,472 weight values               |
| `W @ features` in one line | `layers.Dense(2304)(x)` in one line   |

**The operation is identical. Only the shape changes.**

**Missing piece:** We can compute $y = Wx + b$ forward, but gradient descent needs $\frac{\partial L}{\partial W}$ — how does every single weight in $W$ affect the final loss? Computing that for a matrix with millions of entries requires the chain rule applied systematically.

---

#### Your turn — matrix rows control outputs independently

```python
# CHANGE W so that row 0 weights angle (col 0) heavily and ignores speed (col 1),
#    and row 1 does the opposite. Then double the angle feature and watch which output moves.
W_test = np.array([[3.0, 0.1],   # row 0: cares about angle, ignores speed
                   [0.1, 3.0]])  # row 1: cares about speed, ignores angle
b_test = np.array([0.0, 0.0])

features_normal = np.array([0.4, 0.8])
features_double = np.array([0.8, 0.8])  # double the angle feature only

out_normal = W_test @ features_normal + b_test
out_double = W_test @ features_double + b_test

print(f"Normal features {features_normal}: output = {out_normal.round(3)}")
print(f"Doubled angle  {features_double}: output = {out_double.round(3)}")
print(f"Delta: {(out_double - out_normal).round(3)}")
```

---

## Part 5 — Start at the Net and Investigate Backward

The ball hits the back net **0.232 m too high**.

Before writing any derivative, reason like an investigator:

1. The impact is high, so the miss is positive.
2. Near this shot, making the angle steeper raises the impact.
3. Therefore a steeper angle would make the positive miss worse.
4. The next update should lower the angle.

That is the conclusion backpropagation must produce numerically.

The forward pass stored the chain of consequences:

**launch angle → flight → net height → target miss → wrongness**

The backward pass walks that chain in reverse. At each link it asks:

> “If this earlier value changed a little, how much would the final wrongness change?”

Watch the green signal travel from the impact back to the launch angle. The ball is not travelling backward in time; **responsibility is travelling backward through the recorded calculation**.

<details>
<summary><strong>Name the pattern: chain rule</strong></summary>

For the target branch:

$$\frac{dL}{d\theta}=\frac{dL}{d\text{miss}}\times\frac{d\text{miss}}{d\text{impact height}}\times\frac{d\text{impact height}}{d\theta}$$

At the animated shot, the local factors multiply to $+0.039809$ loss per degree. Positive means “a larger angle raises wrongness,” confirming the verbal prediction.

</details>

> **The intuition:** forward mode tells the story of consequences. Backward mode asks who was responsible. The gradient returned to an earlier value is its share of the final wrongness.

![Cinematic reverse-pass sequence that freezes the impact and carries responsibility back to the launch angle](images/backprop-free-kick.gif)

**Try to predict the final instruction before it appears:** because the ball is high and a steeper angle raises it, the update must lower the angle.

In [ ]:
# Part 5: explicit reverse-mode chain rule on the same free-kick loss
def dh_dtheta(x, theta_deg):
    """Derivative of ball height with respect to angle measured in degrees."""
    theta_rad = np.radians(theta_deg)
    sec2 = 1.0 / np.cos(theta_rad) ** 2
    a = g * x**2 / (2 * v0**2)
    dh_dradians = x * sec2 - 2 * a * sec2 * np.tan(theta_rad)
    return dh_dradians * np.pi / 180.0


def analytic_kick_gradient(theta_deg):
    """Backpropagate through every currently active loss branch."""
    h_wall = ball_height(WALL_X, theta_deg)
    h_goal = ball_height(GOAL_X, theta_deg)
    h_net = ball_height(NET_X, theta_deg)

    wall_gap = max(0.0, WALL_H + BALL_RADIUS - h_wall)
    goal_low_gap = max(0.0, GOAL_BOTTOM - h_goal)
    goal_high_gap = max(0.0, h_goal - GOAL_TOP)
    net_error = h_net - TARGET_H

    gradient = 0.5 * net_error * dh_dtheta(NET_X, theta_deg)
    if wall_gap > 0:
        gradient -= 2 * wall_gap * dh_dtheta(WALL_X, theta_deg)
    if goal_low_gap > 0:
        gradient -= 4 * goal_low_gap * dh_dtheta(GOAL_X, theta_deg)
    if goal_high_gap > 0:
        gradient += 4 * goal_high_gap * dh_dtheta(GOAL_X, theta_deg)
    return gradient


trace_theta = final_angle + 0.65
trace_net_height = ball_height(NET_X, trace_theta)
trace_error = trace_net_height - TARGET_H
local_dloss_derror = 0.5 * trace_error
local_dheight_dtheta = dh_dtheta(NET_X, trace_theta)
chain_gradient = analytic_kick_gradient(trace_theta)
numerical_gradient = (kick_loss(trace_theta + 1e-4) - kick_loss(trace_theta - 1e-4)) / 2e-4

print(f"forward angle:                 {trace_theta:.6f}°")
print(f"forward back-net impact:       {trace_net_height:.6f}m")
print(f"forward target error:          {trace_error:+.6f}m")
print()
print(f"backward dL/de:                {local_dloss_derror:+.6f}")
print(f"backward dh_net/dtheta:        {local_dheight_dtheta:+.6f} m/degree")
print(f"chain-rule dL/dtheta:          {chain_gradient:+.6f}")
print(f"finite-difference dL/dtheta:   {numerical_gradient:+.6f}")
print(f"match within 1e-6:             {abs(chain_gradient - numerical_gradient) < 1e-6}")
print()
print("Because dL/dtheta is positive, the next optimizer update reduces theta.")

#### The backward answer matches the human answer

The reverse pass returned a positive angle responsibility: **+0.039809 loss per degree**.

In words:

> A larger angle would make this high impact worse. Lower the angle.

The finite-difference check reached the same number, which tells us the backward explanation belongs to the real forward calculation rather than to a disconnected diagram.

Frameworks such as TensorFlow and PyTorch automate the bookkeeping:

1. remember the important values while moving forward,
2. begin at the final wrongness,
3. pass sensitivity backward through each operation,
4. combine responsibility when several paths reach the same parameter.

#### Your turn — when two mistakes blame the same angle

Set the angle to 16°. Now the wall path and target path both contribute. Predict their directions before printing them. Backpropagation adds those contributions because the same launch angle caused both consequences.

<details>
<summary><strong>Why this scales</strong></summary>

Finite differences rerun the whole model after perturbing each parameter. Reverse-mode autodiff traverses the recorded graph once and reuses intermediate values. That is why millions of gradients can be computed in one backward pass.

</details>

---

## Part 6 — A Perfect Plan Is Not Perfect Execution

Gradient descent found where to aim. It did not make the striker mechanically perfect.

Suppose the intended angle is the centre of the legal window. Real attempts scatter around it:

- a consistent striker produces a tight cluster,
- an inconsistent striker produces a wide cluster,
- only the attempts inside the legal angle window score.

Probability answers a new question:

> **If I repeat this plan many times, what fraction of executions should succeed?**

The legal window is narrow: about **18.44° to 21.86°**. With 3° of typical variation, much of the attempt distribution falls outside it.

#### Predict first

What scoring probability do you expect?

- **(a)** Above 90%
- **(b)** Around 40–45%
- **(c)** Below 10%

<details>
<summary><strong>Name the pattern: probability mass</strong></summary>

We model the executed angle as a Gaussian centred on the intended aim. Scoring probability is the area of that distribution inside the legal interval:

$$P(\text{score})=P(18.44°<\Theta<21.86°)$$

The direct ML bridge is negative log-likelihood: assigning low probability to what actually happened produces a large loss. Softmax handles discrete classes; the Gaussian here handles one continuous noisy angle.

</details>

> **Two different jobs:** optimisation chooses the intended action; probability describes how execution spreads around that intention. “Where should I aim?” and “How reliably can I do it?” are not the same question.

In [ ]:
# Part 6: probability of geometric scoring given noisy execution
from scipy import stats

# Sample angles and keep only those that clear the wall and cross fully inside the goal frame
scoreable_angles = [
    angle for angle in np.linspace(5, 60, 20_000)
    if (
        ball_height(WALL_X, angle) > WALL_H + BALL_RADIUS
        and GOAL_BOTTOM < ball_height(GOAL_X, angle) < GOAL_TOP
    )
]

if not scoreable_angles:
    raise RuntimeError("No launch angle satisfies the physical constraints")

theta_lo = min(scoreable_angles)
theta_hi = max(scoreable_angles)
probability_aim = (theta_lo + theta_hi) / 2

print(f"Legal angle window: [{theta_lo:.2f}°, {theta_hi:.2f}°]")
print(f"Window width: {theta_hi - theta_lo:.2f}°")
print(f"Aim that maximises symmetric Gaussian mass: {probability_aim:.2f}°")
print(f"Target-height optimizer solution:           {final_angle:.2f}°")
print("These differ because they optimise different objectives.\n")

sigma = 3.0
normal = stats.norm(loc=probability_aim, scale=sigma)
prob_score = normal.cdf(theta_hi) - normal.cdf(theta_lo)
print(f"With sigma={sigma}° execution variability: P(score)={prob_score:.1%}")
print("Prediction check: answer (b).\n")

print("P(scoring) vs execution precision:")
for sigma_test in [1.0, 2.0, 3.0, 5.0, 8.0, 10.0]:
    distribution = stats.norm(loc=probability_aim, scale=sigma_test)
    probability = distribution.cdf(theta_hi) - distribution.cdf(theta_lo)
    print(f"  sigma={sigma_test:4.1f}°: P(score)={probability:.1%}")

### Imagine 100 repeated kicks

The aim stays fixed. Only execution varies.

| Typical variation | Rough number scoring out of 100 | What the crowd of kicks looks like |
| ---: | ---: | --- |
| 1° | 91 | A tight cluster, mostly inside the window |
| 2° | 61 | The edges begin to spill out |
| 3° | 43 | More than half now miss the narrow window |
| 5° | 27 | A broad spread with few legal outcomes |
| 8° | 17 | Intention is overwhelmed by inconsistency |

The probability is not predicting the fate of one particular kick. It describes the long-run share of a repeated process.

#### The deeper intuition

Improving the **mean** moves the centre of the cluster. Improving **consistency** tightens the cluster. A system can need either kind of improvement — or both.

> Optimisation asks where the centre should be. Probability asks how much of the spread lands somewhere acceptable.

#### What just happened — and what's missing

The shaded area is the probability that a noisy continuous angle lands inside the physically legal interval. With σ=3°, it is about **43.1%**. Reducing σ to 1° raises it above 91%; improving consistency can matter as much as improving the intended aim.

The negative log converts probability into an optimisation objective:

$$L_{NLL}=-\log P(\text{observed outcome})$$

- if $P=0.9$, then $L\approx0.105$,
- if $P=0.1$, then $L\approx2.303$.

That is the direct connection to cross-entropy: for a one-hot class label, cross-entropy is the negative log probability assigned to the correct class. The Gaussian and softmax distributions have different shapes and domains, but maximum likelihood gives both the same "make observed outcomes less surprising" training principle.

---

#### Your turn — cost of poor technique

```python
# CHANGE sigma_test to 8.0 and 1.0.
sigma_test = 8.0

distribution_test = stats.norm(loc=probability_aim, scale=sigma_test)
probability_test = distribution_test.cdf(theta_hi) - distribution_test.cdf(theta_lo)
print(f"sigma={sigma_test}°: P(scoring)={probability_test:.1%}")
print(f"negative log-likelihood of scoring={-np.log(probability_test + 1e-12):.3f}")
```

This calculation does not claim every football error is Gaussian. It demonstrates how an uncertainty assumption becomes a probability and then a trainable loss.

---

## Summary — One Mental Model for Training

| Part | Tool | Free-kick result | ML connection | Working intuition |
| ---- | ---- | ---------------- | ------------- | ----------------- |
| 1 | Vectors + dot products | 5° directional difference gives alignment 0.9962 | Dense layers and attention scores | A dot product asks how strongly two directions overlap |
| 2 | Derivatives | 25° shot peaks near 15.6 m and crosses over the bar | Local parameter sensitivity | A derivative is a speedometer for change, not a picture of the whole curve |
| 3 | Gradient descent | 35° → 19.13°; loss 45.68 → 0.000018 | Optimizer training loop | Measure the miss, compute blame, step opposite the blame |
| 4 | Matrices | Multiple features transformed together | `W @ x + b` in every linear layer | A matrix is many coordinated dot products |
| 5 | Chain rule | Net error traced back to $dL/d\theta=0.039809$ | Reverse-mode autodiff / backpropagation | Forward computes consequences; backward assigns sensitivity |
| 6 | Gaussian probability | Legal window 18.44°–21.86°; P(score)=43.1% at σ=3° | Likelihood and uncertainty modelling | Optimisation chooses an intention; probability describes noisy execution |

### The training loop you should now be able to see

1. **Parameters create a forward world.** Here one angle creates time, position, velocity, and impact. In a model, weights create activations and predictions.
2. **A loss names what "wrong" means.** A bad loss can reward nonsense — as the original crossbar-only loss did for a ball that landed before goal.
3. **Calculus measures responsibility.** The gradient says how much the loss changes per unit change in each parameter.
4. **The optimizer acts on that measurement.** It does not understand football or language; it follows the signed sensitivities.
5. **Probability separates best intention from reliable execution.** A deterministic optimum is not the same as a high-probability outcome.

That cycle — **forward values → loss → backward sensitivities → update** — is the calculus engine behind model training, control systems, trajectory optimisation, inverse graphics, and much of modern scientific computing.

In [ ]:
# Closing decision: combine optimization, geometry, and uncertainty
final_angle = history[-1]
final_loss = kick_loss(final_angle)
final_wall = ball_height(WALL_X, final_angle)
final_goal = ball_height(GOAL_X, final_angle)
final_net = ball_height(NET_X, final_angle)

print("=" * 64)
print("  CLOSING DECISION — WHAT DID CALCULUS BUY US?")
print("=" * 64)
print(f"  Optimized launch angle:       {final_angle:.3f}°")
print(f"  Ball centre over wall:        {final_wall:.3f}m  (need > {WALL_H + BALL_RADIUS:.2f}m)")
print(f"  Ball centre through goal:     {final_goal:.3f}m  (legal [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m)")
print(f"  Back-net impact:              {final_net:.3f}m  (target {TARGET_H:.2f}m)")
print(f"  Target error:                 {final_net - TARGET_H:+.4f}m")
print(f"  Physical loss:                {final_loss:.8f}")
print()
print(f"  Legal geometric window:       [{theta_lo:.2f}°, {theta_hi:.2f}°]")
print(f"  Probability-maximizing aim:   {probability_aim:.2f}°")
print(f"  P(scoring) with sigma=3°:     {prob_score:.1%}")
print()
print("  The deterministic optimizer found where to aim for the chosen target.")
print("  Probability then quantified how execution noise changes the decision.")
print("  Neural-network training uses the same four stages:")
print("  forward values -> loss -> backward gradients -> parameter update.")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

- Vectors and dot products — directional alignment and the role of $W\cdot x$
- Derivatives — analytic peak location and local slope interpretation
- Loss design — piecewise physical constraints plus a continuously active target term
- Gradient descent — 36 measured updates from a high miss to a target-height shot
- Matrix transforms — parallel feature mixing and connection to linear layers
- Chain rule — explicit reverse-mode gradient, including sum-over-active-paths behavior
- Gradient verification — analytic result checked against central finite differences
- Gaussian probability — legal-window probability under execution noise
- High-resolution animations — forward telemetry, optimization loop, and reverse sensitivity flow

### Tier 2 — Explained but Not Fully Demonstrated

- **Partial derivatives** — the one-parameter derivative generalises to gradients over many parameters
- **Reverse-mode autodiff internals** — the notebook performs the local products explicitly but does not build a general autodiff engine
- **Convexity** — the selected one-dimensional region is well behaved; neural-network landscapes are high-dimensional and non-convex

### Tier 3 — Named but Out of Scope

- **Hessians** — second-order curvature and Newton-style methods
- **Taylor series** — local polynomial approximation
- **Information theory** — entropy and KL divergence beyond the negative-log-likelihood bridge
- **Aerodynamic knuckleball forces** — drag, lift, spin, and turbulent wake dynamics; the projectile model here is ballistic

---

## When to Use What — From This Notebook

| Situation | Tool | Why |
| --------- | ---- | --- |
| "How aligned are two feature vectors?" | Dot product | Attention score $Q\cdot K$ and linear prediction both use alignment-weighted sums |
| "How fast is this output changing here?" | Derivative | It measures local sensitivity with units, such as metres per degree |
| "Which parameter change reduces loss?" | Gradient descent | Update = parameter − learning rate × gradient |
| "What does a linear layer do?" | Matrix multiply $Wx+b$ | It applies many learned dot products in parallel |
| "How does a final error depend on an early value?" | Chain rule | Multiply local sensitivities backward; sum when paths merge |
| "Does my gradient correspond to the real program?" | Finite-difference check | A small numerical perturbation is a valuable implementation test |
| "How reliable is a decision under continuous noise?" | Probability model | Integrate probability mass over the acceptable outcome region |
| "How do probabilities become a trainable objective?" | Negative log-likelihood | $-\log p$ penalises assigning low probability to observed outcomes |

---

## What's Next

Every tool in this notebook appears again in prerequisite 01, now applied to data and learned parameters:

| Tool from here | Reappears as |
| -------------- | ------------ |
| Dot product | Feature × weight in a linear model; query × key in attention |
| Derivative / gradient | Sensitivity of data loss to every trainable parameter |
| Gradient descent | Optimizer updates, including momentum and adaptive learning rates |
| Matrix multiply | Every dense layer and transformer projection |
| Chain rule | Automatic differentiation through all model operations |
| Gradient check | A debugging technique for custom losses and layers |
| Probability / likelihood | Predictive uncertainty, softmax class probabilities, and cross-entropy |

The durable mental model is not "a ball rolls downhill." It is:

> **Run the world forward. Measure what was wrong. Trace sensitivity backward. Update the causes.**

The same pattern powers the next notebook's regression and classification models, then scales to deep networks with many parameters and branching computation graphs.

→ **Next:** [`../01-ml-basics/ml-basics.ipynb`](../01-ml-basics/ml-basics.ipynb) — linear regression and classification from scratch, on real data, using the tools you just built.